## 탐지 데이터·모델 연결 검사

공통 표본 Train 2장·Validation 2장으로 데이터가 SSDLite 모델에 정상적으로 전달되는지 확인합니다.

준비
- 공유받은 smoke_test.zip을 풀어 datasets/smoke_test/에 배치합니다.
- 팀 저장소의 학습 환경을 Notebook 커널로 선택합니다.
- 아래 셀을 위에서부터 실행합니다.
검사 내용
- 이미지·라벨·분할 정보 확인
- 모델 입력 형식 확인
- Train 배치의 손실 계산·역전파·가중치 갱신 1회
- Validation 배치의 예측 형식 확인

통과 기준

마지막 보고서의 전체 결과와 모델 연결이 모두 passed이면 연결 검사가 완료됩니다. 이후 START_DETECTOR_DATA.ipynb에서 전체 학습 데이터를 준비합니다.

전체 학습 데이터는 이 검사에 필요하지 않으며, 모델 정확도를 평가하는 검사는 아닙니다.

## 1. 저장소와 실행 환경 확인

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "detector").is_dir() and (p / "detector_data").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("저장소 안에서 Notebook을 실행하세요.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("저장소:", project_root)
print("Python:", sys.executable)


## 2. 표본 파일 확인

이미지·라벨의 해시와 원래 배정된 표본 ID를 확인합니다. 모델은 실행하지 않습니다.
예전 코드 해시는 실행 조건으로 사용하지 않습니다.

In [ ]:
subprocess.run([sys.executable, "-m", "detector_data", "smoke-check"],
               cwd=project_root, check=True)


## 3. 모델 연결 검사

Train 2장: 손실 계산 → 역전파 → 가중치 갱신 1회

Validation 2장: 예측 개수·형식·유효한 수치 확인

`[3/3 통과]`와 `[통과] 검사 완료`를 확인하세요.
CPU가 기본이며, 전체 학습이나 가중치 파일 저장은 하지 않습니다.

In [ ]:
report_path = project_root / "outputs/detector_data/shared_smoke_report.json"
subprocess.run(
    [sys.executable, "-m", "detector_data", "smoke", "--report", str(report_path)],
    cwd=project_root, check=True,
)


## 4. 검사 결과 확인

`전체 결과`와 `모델 연결`이 모두 `passed`이면 검사 통과입니다.

보고서 저장 위치: `outputs/detector_data/shared_smoke_report.json`

In [ ]:
report = json.loads(report_path.read_text(encoding="utf-8"))
print("보고서:", report_path)
print("전체 결과:", report["status"])
print("모델 연결:", report.get("model_step", {}).get("status", "실행 안 됨"))
print(json.dumps(report, ensure_ascii=False, indent=2))
